In [ ]:
import pandas as pd
import requests
import io
from google.colab import files

print("¡Bibliotecas cargadas con éxito!")

Bibliotecas carregadas com sucesso!


In [ ]:
url_chs = "https://www.chsegura.es/es/cuenca/redes-de-control/estadisticas-hidrologicas/estado-de-embalses/"

print("Leyendo tablas del sitio web de CHS...")
tabelas = pd.read_html(url_chs, decimal=',', thousands='.')

df_atual = tabelas[3][['Unnamed: 0', 'Total (hm³)']].copy()
df_atual = df_atual.rename(columns={'Unnamed: 0': 'Data', 'Total (hm³)': 'Volume_hm3'})

df_atual['Data'] = df_atual['Data'].str.split(' ').str[0]
df_atual['Data'] = pd.to_datetime(df_atual['Data'], format='%d/%m/%Y')

print("Datos recientes estructurados:")
print(df_atual.head())

Lendo tabelas do site da CHS...
Dados recentes estruturados:
        Data  Volume_hm3
0 2026-07-30         631
1 2026-07-31         630
2 2026-08-01         629
3 2026-08-02         627
4 2026-08-03         626


In [ ]:
# Upload del Historial
print("Por favor, suba el archivo 'segura_total_short.csv':")

arquivo_upado = files.upload()

nome_arquivo = list(arquivo_upado.keys())[0]

print(f"\nArchivo '{nome_arquivo}' recibido con éxito!")

Por favor, clique no botão abaixo e faça o upload do arquivo 'segura_total_short.csv':


Saving segura_total_short.csv to segura_total_short.csv

Arquivo 'segura_total_short.csv' recebido com sucesso na memória do Colab!


In [ ]:
df_historico = pd.read_csv(io.BytesIO(arquivo_upado[nome_arquivo]))

df_historico.columns = df_historico.columns.str.lower()

if 'fecha' in df_historico.columns:
    df_historico['fecha'] = pd.to_datetime(df_historico['fecha'])
    df_historico = df_historico.sort_values(by='fecha')
    df_historico = df_historico.rename(columns={'fecha': 'Data'})

print("Serie histórica procesada y lista para el modelado:")
print(df_historico.head())

Série histórica processada e pronta para modelagem:
        Data  week  media 5 años  media 10 años  2025_2026  2024_2025  \
0 2026-09-29    40           342            319      185.0        184   
1 2026-10-06    41           336            314      195.0        179   
2 2026-10-13    42           329            308      206.0        172   
3 2026-10-20    43           322            302      212.0        166   
4 2026-10-27    44           318            299      219.0        168   

   2023_2024  2022_2023  2021_2022  2020_2021  ...  1997_1998  1996_1997  \
0        278        387        426        436  ...        295        168   
1        270        385        417        428  ...        301        164   
2        261        385        406        420  ...        300        159   
3        254        383        398        411  ...        296        154   
4        248        379        392        401  ...        292        154   

   1995_1996  1994_1995  1993_1994  1992_1993  1991_

In [ ]:
url_ine = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/2074?nult=60"

print("Consultando a API do INE...")
resposta = requests.get(url_ine)

if resposta.status_code == 200:
    dados_json = resposta.json()
    registros = []
    provincias_alvo = ['Murcia', 'Alicante', 'Almería']

    for serie in dados_json:
        nome_serie = serie.get('Nombre', '')


        if any(prov in nome_serie for prov in provincias_alvo):
            if 'Data' in serie:
                for dado in serie['Data']:
                    registros.append({
                        'Indicador': nome_serie,
                        'Ano': dado.get('Anyo'),
                        'Mes': dado.get('FK_Periodo'),
                        'Valor': dado.get('Valor')
                    })

    df_turismo = pd.DataFrame(registros).dropna(subset=['Valor'])
    print("\nDados socioeconômicos extraídos:")
    print(df_turismo.head())
else:
    print(f"Falha na API. Código: {resposta.status_code}")

Consultando a API do INE...

Dados socioeconômicos extraídos:
                   Indicador   Ano  Mes     Valor
0  Almería. Viajeros. Total.  2026    6  215249.0
1  Almería. Viajeros. Total.  2026    5  151791.0
2  Almería. Viajeros. Total.  2026    4  115549.0
3  Almería. Viajeros. Total.  2026    3   85105.0
4  Almería. Viajeros. Total.  2026    2   58574.0


In [ ]:
colunas_anos = [col for col in df_historico.columns if col.startswith('19') or col.startswith('20')]

df_agua_long = pd.melt(
    df_historico,
    id_vars=['Data', 'week'],
    value_vars=colunas_anos,
    var_name='Ano_Hidrologico',
    value_name='Volume_Agua_hm3'
)

df_agua_long['Ano'] = df_agua_long['Ano_Hidrologico'].str[:4].astype(int)

df_agua_anual = df_agua_long.groupby('Ano')['Volume_Agua_hm3'].mean().reset_index()

df_turismo_anual = df_turismo.groupby(['Indicador', 'Ano'])['Valor'].sum().reset_index()

df_master = pd.merge(df_turismo_anual, df_agua_anual, on='Ano', how='inner')

print("\nTabela Master (Turismo x Água) gerada com sucesso!")
print(df_master.head(10))

df_master.to_csv('tabela_master_seca_turismo.csv', index=False)


Tabela Master (Turismo x Água) gerada com sucesso!
                                           Indicador   Ano       Valor  \
0    Alicante. Pernoctaciones. Residentes en España.  2021   4380708.0   
1    Alicante. Pernoctaciones. Residentes en España.  2022   7839888.0   
2    Alicante. Pernoctaciones. Residentes en España.  2023   7636985.0   
3    Alicante. Pernoctaciones. Residentes en España.  2024   7713673.0   
4    Alicante. Pernoctaciones. Residentes en España.  2025   7457737.0   
5  Alicante. Pernoctaciones. Residentes en el ext...  2021   2416319.0   
6  Alicante. Pernoctaciones. Residentes en el ext...  2022   8240105.0   
7  Alicante. Pernoctaciones. Residentes en el ext...  2023   9657950.0   
8  Alicante. Pernoctaciones. Residentes en el ext...  2024  10837541.0   
9  Alicante. Pernoctaciones. Residentes en el ext...  2025  11474469.0   

   Volume_Agua_hm3  
0       430.076923  
1       377.326923  
2       235.653846  
3       271.480769  
4       472.162791  
5      

In [ ]:
print(df_historico.columns)
df_historico.head(3)

Index(['Data', 'week', 'media 5 años', 'media 10 años', '2025_2026',
       '2024_2025', '2023_2024', '2022_2023', '2021_2022', '2020_2021',
       '2019_2020', '2018_2019', '2017_2018', '2016_2017', '2015_2016',
       '2014_2015', '2013_2014', '2012_2013', '2011_2012', '2010_2011',
       '2009_2010', '2008_2009', '2007_2008', '2006_2007', '2005_2006',
       '2004_2005', '2003_2004', '2002_2003', '2001_2002', '2000_2001',
       '1999_2000', '1998_1999', '1997_1998', '1996_1997', '1995_1996',
       '1994_1995', '1993_1994', '1992_1993', '1991_1992', '1990_1991',
       '1989_1990', '1988_1989'],
      dtype='object')


,Data,week,media 5 años,media 10 años,2025_2026,2024_2025,2023_2024,2022_2023,2021_2022,2020_2021,...,1997_1998,1996_1997,1995_1996,1994_1995,1993_1994,1992_1993,1991_1992,1990_1991,1989_1990,1988_1989
0,2026-09-29,40,342,319,185.0,184,278,387,426,436,...,295,168,84,76,87,107,112,237,212,112
1,2026-10-06,41,336,314,195.0,179,270,385,417,428,...,301,164,85,81,93,96,111,232,209,100
2,2026-10-13,42,329,308,206.0,172,261,385,406,420,...,300,159,87,90,92,101,110,229,206,99


In [ ]:
import pandas as pd
import io
from google.colab import files

print("Sube el archivo CSV del INE:")
arquivo_agr = files.upload()
nome_arq_agr = list(arquivo_agr.keys())[0]

df_agr = pd.read_csv(io.BytesIO(arquivo_agr[nome_arq_agr]), sep=';', encoding='latin1')
df_agr.columns = ['Total_Nacional', 'Comunidad', 'Origen_Agua', 'Ano', 'Volume_Agua_Agricola']

df_agr['Comunidad'] = df_agr['Comunidad'].replace('Murcia, Región de', 'Región de Murcia')

provincias_alvo = ['Andalucía', 'Comunitat Valenciana', 'Región de Murcia']
df_agr = df_agr[df_agr['Comunidad'].isin(provincias_alvo)].copy()

df_agr['Volume_Agua_Agricola'] = (
    df_agr['Volume_Agua_Agricola']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
)
df_agr['Volume_Agua_Agricola'] = pd.to_numeric(df_agr['Volume_Agua_Agricola'], errors='coerce')
df_agr['Ano'] = df_agr['Ano'].astype(str).str.extract(r'(\d{4})')[0].astype(int)
df_agr_anual = df_agr.groupby(['Comunidad', 'Ano'])['Volume_Agua_Agricola'].sum().reset_index()

print("\nDatos limpios y filtrados:")
print(df_agr_anual['Comunidad'].unique())

df_master_final = pd.merge(df_agr_anual, df_agua_anual, on='Ano', how='inner')
df_master_final.to_csv('tabela_master_seca_agricultura.csv', index=False)
print("\nPipeline de ETL da Agricultura concluído!")

Faça o upload do arquivo CSV do INE (Agricultura):


Saving 02001(3).csv to 02001(3).csv

Dados Agrícolas limpos e filtrados (Múrcia salva com sucesso!):
['Andalucía' 'Comunitat Valenciana' 'Región de Murcia']

Pipeline de ETL da Agricultura concluído!


In [ ]:
import pandas as pd
import requests
from google.colab import files

print("Conectando à API do Open-Meteo (Acesso livre e sem API Key)...")

coordenadas = {
    'Comunitat Valenciana': {'lat': 38.3452, 'lon': -0.4810},
    'Región de Murcia': {'lat': 37.9922, 'lon': -1.1307},
    'Andalucía': {'lat': 36.8340, 'lon': -2.4637}
}

lista_clima = []

for comunidad, coords in coordenadas.items():
    print(f"Baixando histórico de chuva para {comunidad}...")
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={coords['lat']}&longitude={coords['lon']}&start_date=2010-01-01&end_date=2023-12-31&daily=precipitation_sum&timezone=Europe/Madrid"

    resposta = requests.get(url)

    if resposta.status_code == 200:
        dados = resposta.json()
        df_temp = pd.DataFrame({
            'Data': dados['daily']['time'],
            'Precipitacao_mm': dados['daily']['precipitation_sum']
        })

        df_temp['Data'] = pd.to_datetime(df_temp['Data'])
        df_temp['Ano'] = df_temp['Data'].dt.year
        df_anual = df_temp.groupby('Ano')['Precipitacao_mm'].sum().reset_index()
        df_anual['Comunidad'] = comunidad

        lista_clima.append(df_anual)
    else:
        print(f"Erro ao conectar com {comunidad}: HTTP {resposta.status_code}")

if len(lista_clima) > 0:
    df_clima = pd.concat(lista_clima, ignore_index=True)
    print("\nBase de dados meteorológica estruturada com sucesso!")
    df_master_completa = pd.merge(df_master_final, df_clima, on=['Ano', 'Comunidad'], how='inner')

    print("\nTabela Master Definitiva (Causas Naturais + Causas Humanas + Escassez) gerada!")
    print(df_master_completa.head())
    df_master_completa.to_csv('tabela_master_seca_completa.csv', index=False)
    print("\nDownload da tabela final iniciado...")
    files.download('tabela_master_seca_completa.csv')
else:
    print("\nFalha na extração. A tabela de clima está vazia.")

Conectando à API do Open-Meteo (Acesso livre e sem API Key)...
Baixando histórico de chuva para Comunitat Valenciana...
Baixando histórico de chuva para Región de Murcia...
Baixando histórico de chuva para Andalucía...

Base de dados meteorológica estruturada com sucesso!

Tabela Master Definitiva (Causas Naturais + Causas Humanas + Escassez) gerada!
   Comunidad   Ano  Volume_Agua_Agricola  Volume_Agua_hm3  Precipitacao_mm
0  Andalucía  2010               8383752       740.711538            319.4
1  Andalucía  2011               8312348       602.653846            158.9
2  Andalucía  2012               7672224       687.788462            190.3
3  Andalucía  2013               7963536       775.480769            169.1
4  Andalucía  2014               9038612       658.576923            145.7

Download da tabela final iniciado...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
from google.colab import files

mapeamento_geo = {
    'Comunitat Valenciana': 'Alicante',
    'Andalucía': 'Almería',
    'Región de Murcia': 'Murcia'
}

df_master_completa['Provincia_Alvo'] = df_master_completa['Comunidad'].map(mapeamento_geo)

colunas_atuais = df_master_completa.columns.tolist()
colunas_atuais.remove('Provincia_Alvo')
colunas_reorganizadas = ['Provincia_Alvo'] + colunas_atuais

df_master_completa = df_master_completa[colunas_reorganizadas]

print("Granularidade corrigida! Dados mapeados para as cidades alvo do projeto:")
print(df_master_completa[['Provincia_Alvo', 'Comunidad', 'Ano']].head())

df_master_completa.to_csv('tabela_master_seca_completa_FINAL.csv', index=False)
print("\nIniciando o download do arquivo perfeitamente formatado...")
files.download('tabela_master_seca_completa_FINAL.csv')

Granularidade corrigida! Dados mapeados para as cidades alvo do projeto:
  Provincia_Alvo  Comunidad   Ano
0        Almería  Andalucía  2010
1        Almería  Andalucía  2011
2        Almería  Andalucía  2012
3        Almería  Andalucía  2013
4        Almería  Andalucía  2014

Iniciando o download do arquivo perfeitamente formatado...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
%whos DataFrame

Variable             Type         Data/Info
-------------------------------------------
df_agr               DataFrame        Total_Nacional       <...>n\n[144 rows x 5 columns]
df_agr_anual         DataFrame                   Comunidad <...>018               3300242
df_agua_anual        DataFrame         Ano  Volume_Agua_hm3<...>37  2025       472.162791
df_agua_long         DataFrame               Data  week Ano<...>\n[1976 rows x 5 columns]
df_anual             DataFrame         Ano  Precipitacao_mm<...>         239.7  Andalucía
df_atual             DataFrame            Data  Volume_hm3\<...>n6 2026-08-04         626
df_clima             DataFrame         Ano  Precipitacao_mm<...>9.7             Andalucía
df_historico         DataFrame             Data  week  medi<...>n\n[52 rows x 42 columns]
df_master            DataFrame                             <...>n\n[120 rows x 4 columns]
df_master_completa   DataFrame       Provincia_Alvo        <...>173077            388.6  
df_master_fi

In [ ]:
print("AGRICULTURA:", df_agr_anual.columns.tolist())
print("TURISMO:", df_turismo_anual.columns.tolist())
print("AGUA:", df_agua_anual.columns.tolist())

AGRICULTURA: ['Comunidad', 'Ano', 'Volume_Agua_Agricola']
TURISMO: ['Indicador', 'Ano', 'Valor']
AGUA: ['Ano', 'Volume_Agua_hm3']


In [ ]:
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

data = {
    'Provincia_Alvo': ['Alicante', 'Alicante', 'Almería', 'Almería', 'Murcia', 'Murcia'],
    'Comunidad': ['Comunitat Valenciana', 'Comunitat Valenciana', 'Andalucía', 'Andalucía', 'Región de Murcia', 'Región de Murcia'],
    'Ano': [2022, 2023, 2022, 2023, 2022, 2023],
    'Volume_Agua_hm3': [450.5, 410.2, 320.1, 290.4, 510.8, 480.6],
    'Volume_Agua_Agricola': [1200.5, 1150.3, 980.2, 900.1, 1400.7, 1350.9]
}
df = pd.DataFrame(data)

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "Dados_Consolidados"

for r in dataframe_to_rows(df, index=False, header=True):
    ws.append(r)

header_fill = PatternFill(start_color="2F4F4F", end_color="2F4F4F", fill_type="solid")
header_font = Font(name="Calibri", size=12, bold=True, color="FFFFFF")
thin_border = Border(left=Side(style='thin', color='D3D3D3'),
                     right=Side(style='thin', color='D3D3D3'),
                     top=Side(style='thin', color='D3D3D3'),
                     bottom=Side(style='thin', color='D3D3D3'))

for cell in ws[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center", vertical="center")
    cell.border = thin_border

for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
    for cell in row:
        cell.border = thin_border
        cell.alignment = Alignment(horizontal="center")
        if isinstance(cell.value, float):
            cell.number_format = '#,##0.00'

for col in ws.columns:
    max_length = 0
    column = col[0].column_letter
    for cell in col:
        try:
            if len(str(cell.value)) > max_length:
                max_length = len(str(cell.value))
        except:
            pass
    adjusted_width = (max_length + 4)
    ws.column_dimensions[column].width = adjusted_width

ws.freeze_panes = 'A2'

file_path = "Estrutura_Master_Seca_Levante.xlsx"
wb.save(file_path)
print(f"File saved to {file_path}")

File saved to Estrutura_Master_Seca_Levante.xlsx


In [ ]:
from google.colab import files
files.download('tabela_master_seca_completa_FINAL.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('tabela_master_seca_agricultura.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>